# Gaussian Mixture Model baseline

 Every booking gets a probability of belonging to each cluster, and its label is the most likely one

## 1 Setup and data

In [9]:
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from scipy.spatial.distance import pdist
import os
import joblib
import numpy as np
import pandas as pd

RANDOM_STATE = 42
MODEL_DIR = "model"
os.makedirs(MODEL_DIR, exist_ok=True)
pd.set_option("display.max_columns", None)

In [10]:
data = pd.read_csv("../data/H2_Cleaned_clustering.csv")
X = data.to_numpy()
info = pd.read_csv("../data/H2_notdropped.csv")   # to describe clusters only
print("bookings:", len(X), " model features:", X.shape[1])

bookings: 48990  model features: 12


Silhouette is computed on a fixed sample of 10,000 bookings, the other scores on all bookings

In [11]:
rng = np.random.default_rng(RANDOM_STATE)
S = np.sort(rng.choice(len(X), size=10_000, replace=False))   # the fixed silhouette sample


def evaluate(labels):
    sizes = np.bincount(labels)
    return {"clusters": (sizes > 0).sum(),
            "smallest cluster %": sizes.min() / len(labels) * 100,
            "silhouette": silhouette_score(X[S], labels[S]),
            "davies_bouldin": davies_bouldin_score(X, labels),
            "calinski_harabasz": calinski_harabasz_score(X, labels)}

## 2 Input: 11 rotated components

Every booking has exactly one segment, so the six Segment columns always add up to the same value. That leaves one direction in the 12 columns with no variance at all. A full-covariance GMM has to invert the covariance matrix of every component, and with a zero-variance direction that is impossible.

The fix is to rotate the data onto its 11 principal components that do have variance. Only the empty direction is dropped, so the distances between bookings stay the same.

In [12]:
print("smallest variance in the 12 columns:", round(abs(np.linalg.eigvalsh(np.cov(X.T)).min()), 6))

gmm_pca = PCA(n_components=11, random_state=RANDOM_STATE).fit(X)
Xg = gmm_pca.transform(X)
print("smallest variance in the 11 components:", round(np.linalg.eigvalsh(np.cov(Xg.T)).min(), 3))
print("distances between bookings unchanged:", np.allclose(pdist(X[S[:2000]]), pdist(Xg[S[:2000]])))

smallest variance in the 12 columns: 0.0
smallest variance in the 11 components: 0.001
distances between bookings unchanged: True


## 3 Baseline model

scikit-learn defaults (`covariance_type="full"`, `reg_covar=1e-6`, `init_params="kmeans"`, `n_init=1`)

 The only change is `n_components`: the default is 1, which is not a clustering, so the baseline uses 5 like the K-Means baseline.

In [13]:
baseline = GaussianMixture(n_components=5, random_state=RANDOM_STATE).fit(Xg)
labels = baseline.predict(Xg)
pd.DataFrame([evaluate(labels)], index=["baseline"]).round(3)

,clusters,smallest cluster %,silhouette,davies_bouldin,calinski_harabasz
baseline,5,3.215,0.05,2.532,4028.045


In [14]:
proba = baseline.predict_proba(Xg).max(axis=1)   # probability of each booking's own cluster
pd.DataFrame({"share %": np.bincount(labels) / len(X) * 100,
              "assigned with probability >= 0.999 (%)": [(proba[labels == k] >= 0.999).mean() * 100 for k in range(5)]},
             index=[f"C{k}" for k in range(5)]).round(1)

,share %,assigned with probability >= 0.999 (%)
C0,3.2,100.0
C1,3.8,100.0
C2,31.7,100.0
C3,45.4,100.0
C4,15.9,99.9


* 5 clusters, but very uneven: 45.4% of the bookings in C3 and 3.2% in C0
* Silhouette 0.05 and Davies–Bouldin 2.532
* Almost every booking is assigned with probability ≥ 0.999 (99.9–100% in every cluster)

## 4 Why every booking is assigned with certainty

Many columns take only a few whole-number values. If every booking in a component has the same value in one of them, the component has no spread there. Its density at that value becomes huge so any booking with that value is assigned to it with probability close to 1.

The check: for each component, the share of its bookings that have the component's most common value in each whole-number column. 1.0 means no spread at all.

In [15]:
whole = ["TotalOfSpecialRequests", "IsRepeatedGuest", "TotalGuests", "TotalNights"]
same = info[whole].groupby(labels).agg(lambda s: s.value_counts(normalize=True).iloc[0])
same.index = [f"C{k}" for k in range(5)]
same.round(3)

,TotalOfSpecialRequests,IsRepeatedGuest,TotalGuests,TotalNights
C0,0.439,0.843,0.747,0.526
C1,0.842,1.000,0.678,0.332
C2,1.000,1.000,0.652,0.241
C3,0.654,1.000,0.682,0.260
C4,1.000,1.000,0.570,0.301


* In C1 to C4 every booking has the same `IsRepeatedGuest` value, and in C2 and C4 every booking has the same number of special requests.
* So the baseline sorts bookings by exact whole-number values instead of modelling the shape of the data. The certainty is a side effect of that, not a sign of well separated clusters.

## Save the input and the baseline model

In [16]:
joblib.dump(gmm_pca, f"{MODEL_DIR}/gmm_input_pca.joblib")     
joblib.dump(baseline, f"{MODEL_DIR}/gmm_baseline.joblib")

['model/gmm_baseline.joblib']